In [ ]:
from importlib import metadata
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import tempfile

# AMML_DATA может указывать на каталог с CSV или на исходный ZIP соревнования.
DATA_PATH = os.environ.get("AMML_DATA") or None
if DATA_PATH:
    DATA_PATH = str(Path(DATA_PATH).expanduser().resolve())
OUTPUT = Path(os.environ.get("AMML_OUTPUT", "/kaggle/working")).resolve()
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="giga_ranking_", dir=OUTPUT))
SCRIPTS = RUN / "runtime"
SCRIPTS.mkdir()
FOLD = RUN / "fold_0"
FOLD.mkdir()
MODEL_ID = "ai-sage/Giga-Embeddings-instruct-3B-0826"
REVISION = "b71168088212f0a13688514e4fc288a86106c4e4"
SEED = 993
CONFIG_FILE = FOLD / "config.json"
config = {
    "model_id": MODEL_ID, "model_revision": REVISION, "seed": SEED,
    "job_name": "fold_0", "fold": 0, "job_dir": str(FOLD),
    "triplets_jsonl": str(FOLD / "triplets.jsonl"),
    "validation_csv": str(FOLD / "validation_prepared.csv"), "test_csv": str(RUN / "test_prepared.csv"),
    "epochs": 1, "max_length": 512, "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05,
    "learning_rate": 5e-5, "temperature": 0.1, "hard_negatives": 2, "train_batch_size": 1,
    "gradient_accumulation": 8, "infer_batch_size": 4, "smoke_updates": 5,
    "save_every": 50, "log_every": 10, "instruction": "Given a query, retrieve relevant passages",
}

def write_config():
    CONFIG_FILE.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

write_config()
worker_env = dict(os.environ, OMP_NUM_THREADS="2", TOKENIZERS_PARALLELISM="false",
                  HF_HUB_DISABLE_TELEMETRY="1", PYTHONUNBUFFERED="1")
worker_env.setdefault("NCCL_P2P_DISABLE", "1")
worker_env.setdefault("NCCL_IB_DISABLE", "1")
print(f"{platform.system()} · Python {platform.python_version()} · каталог прогона {RUN}")

## Окружение


In [ ]:
REQUIREMENTS = """--extra-index-url https://download.pytorch.org/whl/cu130
torch==2.11.0+cu130
transformers==5.20.2
peft==0.21.2
accelerate==1.15.0
huggingface-hub==1.33.0
tokenizers==0.23.3
safetensors==0.8.0
numpy==2.3.5
pandas==2.2.3
scipy==1.17.0
scikit-learn==1.8.0
joblib==1.5.3
threadpoolctl==3.6.0
ipykernel==7.4.0
"""

def required_versions(lines):
    return dict(line.split("==", 1) for raw in lines
                if (line := raw.split("#", 1)[0].strip()) and not line.startswith("--"))

expected = required_versions(REQUIREMENTS.splitlines())
# При импорте одного ipynb Kaggle не переносит соседние файлы репозитория.
requirements = Path("requirements.txt")
if requirements.is_file():
    if required_versions(requirements.read_text().splitlines()) != expected:
        raise ValueError("Версии в requirements.txt и в импортированном ноутбуке различаются.")
    requirements = requirements.resolve()
else:
    requirements = SCRIPTS / "requirements.txt"
    requirements.write_text(REQUIREMENTS, encoding="utf-8")

def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# Этот старый дополнительный пакет в исходном образе ломал импорт обычной LoRA.
if installed("torchao") == "0.10.0":
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
if any(installed(name) != version for name, version in expected.items()):
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "-r", str(requirements)], check=True)

# Проверка в новом интерпретаторе исключает модули, ранее загруженные ядром Jupyter.
probe = '''import json
import sys
from importlib.metadata import version
import numpy, pandas, scipy, sklearn, torch, transformers, peft, accelerate, tokenizers, safetensors
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
print(json.dumps({"packages": {k: version(k) for k in sys.argv[1:]},
                  "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))
'''
info = json.loads(subprocess.check_output([sys.executable, "-c", probe, *expected], text=True,
                                          env=worker_env).strip().splitlines()[-1])
if info["packages"] != expected:
    raise RuntimeError(f"Версии после установки отличаются от requirements.txt: {info['packages']}")
if len(info["gpu"]) < 2:
    raise RuntimeError("Для воспроизведения прогона выбери ускоритель GPU T4 ×2.")
print(f"CUDA · {info['gpu'][:2]} · torch {info['packages']['torch']} · "
      f"Transformers {info['packages']['transformers']} · PEFT {info['packages']['peft']}")

## Данные и фолд 0


In [ ]:
(SCRIPTS / 'giga_data.py').write_text(r'''import html
import json
import re
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold

REQUIRED = {
    "id", "query_id", "document_id", "article_id", "query", "title", "text",
}
READ = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")


def load_competition_data(data_path=None, input_root="/kaggle/input"):
    def valid(train, test):
        return REQUIRED.union({"relevance"}).issubset(train.columns) and REQUIRED.issubset(test.columns)

    def directory(path):
        a, b = path / "train.csv", path / "test.csv"
        if not a.is_file() or not b.is_file():
            return None
        if not REQUIRED.union({"relevance"}).issubset(pd.read_csv(a, nrows=0).columns):
            return None
        train, test = pd.read_csv(a, **READ), pd.read_csv(b, **READ)
        return (train, test, str(path)) if valid(train, test) else None

    def archive(path):
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            for name in sorted(names):
                other = str(Path(name).with_name("test.csv"))
                if Path(name).name != "train.csv" or other not in names:
                    continue
                with z.open(name) as f:
                    columns = set(pd.read_csv(f, nrows=0).columns)
                if not REQUIRED.union({"relevance"}).issubset(columns):
                    continue
                with z.open(name) as f:
                    train = pd.read_csv(f, **READ)
                with z.open(other) as f:
                    test = pd.read_csv(f, **READ)
                if valid(train, test):
                    return train, test, f"{path} :: {name}"
        return None

    if data_path:
        path = Path(data_path).expanduser()
        result = directory(path) if path.is_dir() else archive(path)
        if result is None:
            raise ValueError("DATA_PATH: не найдены train.csv и test.csv со столбцами соревнования.")
        return result
    root = Path(input_root)
    found = []
    for path in sorted(root.rglob("train.csv")):
        result = directory(path.parent)
        if result is not None:
            found.append(result)
    if not found:
        for path in sorted(root.rglob("*.zip")):
            try:
                result = archive(path)
            except zipfile.BadZipFile:
                continue
            if result is not None:
                found.append(result)
    if len(found) > 1:
        expected = [x for x in found if len(x[0]) == 31530 and len(x[1]) == 13507]
        if len(expected) == 1:
            found = expected
    if len(found) != 1:
        raise RuntimeError(
            "Укажи DATA_PATH явно. Найденные наборы: " + str([x[2] for x in found])
        )
    return found[0]


def clean_text(value):
    value = str(value)
    for _ in range(2):
        decoded = html.unescape(value)
        if decoded == value:
            break
        value = decoded
    value = unicodedata.normalize("NFC", value)
    value = re.sub(r"[\u0300\u0301\u0340\u0341]", "", value)
    value = re.sub(r"</?[A-Za-z][^>]*>", " ", value)
    value = re.sub(r"[\u200b\u2060\ufeff\u00ad]", "", value)
    return re.sub(r"\s+", " ", value).strip()


def clean_title(value):
    value = clean_text(value)
    value = re.sub(r"\s*[—–-]\s*(?:Википедия|Wikipedia)\s*$", "", value, flags=re.I).strip()
    if value.casefold() in {
        "", "nan", "none", "null", "n/a", "без названия", "без заголовка",
        "нет заголовка", "заголовок отсутствует", "untitled",
    }:
        return ""
    return value


def prepare_frames(train, test):
    for name, frame in [("train", train), ("test", test)]:
        if frame["id"].duplicated().any() or frame["id"].str.strip().eq("").any():
            raise ValueError(f"{name}: id должен быть уникальным и заполненным.")
        if frame["query_id"].str.strip().eq("").any():
            raise ValueError(f"{name}: пустые query_id.")
    labels = pd.to_numeric(train["relevance"], errors="raise")
    if not labels.isin([0, 1]).all():
        raise ValueError("relevance должен содержать 0 и 1.")
    if set(train["id"]).intersection(test["id"]):
        raise ValueError("id пересекаются между train и test.")
    joined = pd.concat([train.assign(_split="train"), test.assign(_split="test")], ignore_index=True)
    for column in ["query", "text"]:
        joined[column] = joined[column].map(clean_text)
    joined["title"] = joined["title"].map(clean_title)
    valid_article = ~joined["article_id"].str.strip().str.lower().isin({"", "-1", "nan", "none", "null"})
    titles = (
        joined.loc[valid_article & joined["title"].ne(""), ["article_id", "title"]]
        .drop_duplicates()
    )
    counts = titles.groupby("article_id")["title"].nunique()
    unambiguous = counts.index[counts.eq(1)]
    mapping = titles.loc[titles["article_id"].isin(unambiguous)].set_index("article_id")["title"].to_dict()
    missing = joined["title"].eq("") & valid_article
    restored = joined.loc[missing, "article_id"].map(mapping).fillna("")
    filled = int(restored.ne("").sum())
    joined.loc[missing, "title"] = restored
    joined["document"] = [
        f"{a}. {b}" if a else b for a, b in zip(joined["title"], joined["text"])
    ]
    if joined["query"].eq("").any() or joined["document"].eq("").any():
        raise ValueError("После очистки есть пустой запрос или документ.")
    train_out = joined.loc[joined["_split"].eq("train")].drop(columns="_split").reset_index(drop=True)
    test_out = joined.loc[joined["_split"].eq("test")].drop(columns=["_split", "relevance"]).reset_index(drop=True)
    train_out["relevance"] = labels.to_numpy(dtype=np.int64)
    for name, frame in [("train", train_out), ("test", test_out)]:
        if frame.groupby("query_id")["query"].nunique().gt(1).any():
            raise ValueError(f"{name}: одному query_id соответствуют разные тексты запроса.")
        if frame.groupby("document_id")["document"].nunique().gt(1).any():
            raise ValueError(f"{name}: разные тексты одного document_id после подготовки.")
    return train_out, test_out, {"restored_titles": filled}


def assign_folds(train, n_folds, seed, folds_path=None):
    if folds_path:
        raw = pd.read_csv(folds_path, dtype=str, keep_default_na=False)
        if "query_id" not in raw.columns and {"id", "fold"}.issubset(raw.columns):
            raw = train[["id", "query_id"]].merge(raw[["id", "fold"]], on="id", how="left", validate="one_to_one")
        if not {"query_id", "fold"}.issubset(raw.columns):
            raise ValueError("Файл разбиения должен содержать query_id,fold или id,fold.")
        raw = raw[["query_id", "fold"]].drop_duplicates()
        if raw["query_id"].duplicated().any():
            raise ValueError("Один query_id попал в несколько фолдов.")
        mapping = raw.set_index("query_id")["fold"]
        result = train["query_id"].map(mapping)
        if result.isna().any() or result.astype(str).eq("").any():
            raise ValueError("В файле разбиения отсутствует часть запросов train.")
        result = pd.to_numeric(result, errors="raise")
        if not result.isin(range(n_folds)).all():
            raise ValueError("Номера фолдов должны быть целыми от 0 до N_FOLDS-1.")
        return result.to_numpy(dtype=np.int64)
    queries = np.array(sorted(train["query_id"].unique()))
    if len(queries) < n_folds:
        raise ValueError("Число запросов меньше числа фолдов.")
    mapping = {}
    splitter = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold, (_, indices) in enumerate(splitter.split(queries)):
        mapping.update({str(q): fold for q in queries[indices]})
    return train["query_id"].map(mapping).to_numpy(dtype=np.int64)


def make_triplets(frame, negative_count=2):
    if frame.groupby(["query_id", "document_id"])["relevance"].nunique().gt(1).any():
        raise ValueError("Противоречивые метки одной пары query_id,document_id.")
    frame = frame.drop_duplicates(["query_id", "document_id"]).reset_index(drop=True)
    vectorizer = TfidfVectorizer(
        lowercase=True, ngram_range=(1, 2), sublinear_tf=True,
        min_df=1, max_features=100000, token_pattern=r"(?u)\b\w+\b",
    )
    documents = vectorizer.fit_transform(frame["document"])
    queries = frame[["query_id", "query"]].drop_duplicates("query_id").set_index("query_id")
    query_matrix = vectorizer.transform(queries["query"])
    query_indices = queries.index.get_indexer(frame["query_id"])
    frame["_hardness"] = np.asarray(
        query_matrix[query_indices].multiply(documents).sum(axis=1)
    ).ravel()
    rows = []
    skipped = 0
    for query_id, group in frame.groupby("query_id", sort=False):
        positives = group.loc[group["relevance"].eq(1)].sort_values("id")
        negatives = group.loc[group["relevance"].eq(0)].sort_values(
            ["_hardness", "id"], ascending=[False, True], kind="mergesort"
        )
        if positives.empty or negatives.empty:
            skipped += 1
            continue
        negatives = negatives.head(negative_count)
        for positive in positives.itertuples(index=False):
            selected = [negatives.iloc[i % len(negatives)] for i in range(negative_count)]
            rows.append({
                "query_id": str(query_id), "query": str(positive.query),
                "positive_id": str(positive.document_id),
                "positive": str(positive.document),
                "negatives": [str(x["document"]) for x in selected],
                "negative_ids": [str(x["document_id"]) for x in selected],
            })
    if not rows:
        raise ValueError("Не удалось составить положительно-отрицательные примеры.")
    return rows, {"triplets": len(rows), "queries_without_both_classes": skipped}


def ndcg10(frame, score_column="prediction"):
    if frame.empty or not np.isfinite(frame[score_column].to_numpy(dtype=float)).all():
        raise ValueError("Пустые или нечисловые предсказания.")
    ordered = frame.assign(id=frame["id"].astype(str)).sort_values(
        ["query_id", score_column, "id"], ascending=[True, False, True], kind="mergesort"
    )
    position = ordered.groupby("query_id", sort=False).cumcount().to_numpy() + 1
    gain = np.exp2(ordered["relevance"].to_numpy(dtype=float)) - 1
    contribution = np.where(position <= 10, gain / np.log2(position + 1), 0)
    dcg = pd.Series(contribution, index=ordered.index).groupby(ordered["query_id"], sort=False).sum()
    discount = 1 / np.log2(np.arange(1, 11) + 1)
    positive_count = frame.groupby("query_id")["relevance"].sum()
    idcg = positive_count.map(lambda n: float(discount[:min(int(n), 10)].sum()))
    return float((dcg.reindex(idcg.index).fillna(0) / idcg.replace(0, np.nan)).fillna(0).mean())


def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
''', encoding='utf-8')

In [ ]:
(SCRIPTS / 'prepare.py').write_text(r'''import json
import sys
from pathlib import Path
from giga_data import load_competition_data, prepare_frames, assign_folds, make_triplets

cfg = json.loads(Path(sys.argv[1]).read_text())
source = sys.argv[2] or None
raw_train, raw_test, origin = load_competition_data(source)
train, test, cleaning = prepare_frames(raw_train, raw_test)
train["fold"] = assign_folds(train, 5, cfg["seed"])
fitting = train.loc[train["fold"].ne(cfg["fold"])].copy()
validation = train.loc[train["fold"].eq(cfg["fold"])].copy()
if set(fitting.query_id) & set(validation.query_id):
    raise ValueError("Запрос пересекается между обучением и валидацией")
triplets, stats = make_triplets(fitting, cfg["hard_negatives"])
with Path(cfg["triplets_jsonl"]).open("w", encoding="utf-8") as stream:
    for row in triplets:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
validation.to_csv(cfg["validation_csv"], index=False)
test.to_csv(cfg["test_csv"], index=False)
print(f"Данные из {origin}. Обучение: {len(fitting)} пар и {fitting.query_id.nunique()} запросов. "
      f"Валидация: {len(validation)} пар и {validation.query_id.nunique()} запросов. "
      f"Тест: {len(test)} пар и {test.query_id.nunique()} запросов.")
print(f"Восстановлены заголовки в {cleaning['restored_titles']} строках, "
      f"сформировано {stats['triplets']} примеров для InfoNCE.")
''', encoding='utf-8')

subprocess.run([sys.executable, "-u", str(SCRIPTS / "prepare.py"), str(CONFIG_FILE), DATA_PATH or ""],
               env=worker_env, check=True)

## Дообучение Giga




In [ ]:
(SCRIPTS / 'download_model.py').write_text(r'''import json
import shutil
import sys
import tempfile
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

cfg = json.loads(Path(sys.argv[1]).read_text())
cache = Path(snapshot_download(cfg["model_id"], revision=cfg["model_revision"], max_workers=2,
                               allow_patterns=["*.json", "*.safetensors", "*.py", "*.txt", "*.model"]))
# Для относительных импортов архитектуры файлы .py должны лежать рядом под исходными именами.
model_dir = Path(tempfile.mkdtemp(prefix="giga_model_"))
for source in cache.rglob("*"):
    if not source.is_file():
        continue
    target = model_dir / source.relative_to(cache)
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.suffix == ".safetensors":
        target.symlink_to(source.resolve(strict=True))
    else:
        shutil.copyfile(source, target)
if not (model_dir / "model.safetensors").is_file():
    raise FileNotFoundError("В снимке Giga отсутствуют веса")
spec = AutoConfig.from_pretrained(str(model_dir), trust_remote_code=True, local_files_only=True)
get_class_from_dynamic_module(spec.auto_map["AutoModel"], str(model_dir), local_files_only=True)
Path(sys.argv[2]).write_text(str(model_dir), encoding="utf-8")
print(f"Модель {cfg['model_id']} · ревизия {cfg['model_revision']}")
''', encoding='utf-8')

model_record = RUN / "snapshot_path.txt"
subprocess.run([sys.executable, "-u", str(SCRIPTS / "download_model.py"), str(CONFIG_FILE), str(model_record)],
               env=worker_env, check=True)
config["model_path"] = model_record.read_text(encoding="utf-8")
write_config()

In [ ]:
(SCRIPTS / 'giga_worker.py').write_text(r'''import argparse
import contextlib
import gc
import json
import math
import os
import random
import shutil
import time
from datetime import timedelta
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, get_peft_model_state_dict
from safetensors.torch import load_file
from torch import nn
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from giga_data import ndcg10, write_json


class Triplets(Dataset):
    def __init__(self, path):
        self.rows = [
            json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


def identity_collate(rows):
    return rows


class Encoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, features):
        hidden = self.backbone(**features).last_hidden_state.float()
        mask = features["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        return F.normalize(pooled, p=2, dim=-1)


def load_backbone(cfg, device, adapter_path=None, trainable=True):
    backbone = AutoModel.from_pretrained(
        cfg["model_path"], trust_remote_code=True,
        dtype=torch.float16, attn_implementation="sdpa", low_cpu_mem_usage=True,
    )
    backbone.config.use_cache = False
    if adapter_path:
        backbone = PeftModel.from_pretrained(
            backbone, adapter_path, is_trainable=trainable,
            autocast_adapter_dtype=True,
        )
    else:
        targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
        present = {name.rsplit(".", 1)[-1] for name, _ in backbone.named_modules()}
        missing = sorted(set(targets) - present)
        if missing:
            raise RuntimeError(f"В модели не найдены LoRA-модули: {missing}")
        backbone = get_peft_model(backbone, LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION, r=cfg["lora_rank"],
            lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
            bias="none", target_modules=targets,
        ))
    if trainable:
        for parameter in backbone.parameters():
            if parameter.requires_grad:
                parameter.data = parameter.data.float()
        backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        if not any(getattr(module, "gradient_checkpointing", False) for module in backbone.modules()):
            raise RuntimeError("Gradient checkpointing не включился.")
    return Encoder(backbone).to(device)


def query_input(text, cfg):
    return f'Instruct: {cfg["instruction"]}\nQuery: {text}'


def tokenize(texts, tokenizer, cfg, device):
    features = tokenizer(
        texts, padding=True, truncation=True, max_length=cfg["max_length"],
        return_tensors="pt", return_token_type_ids=False,
    )
    return {name: value.to(device) for name, value in features.items()}


def save_adapter(encoder, destination, cfg, updates):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    encoder.backbone.save_pretrained(destination, safe_serialization=True)
    metadata_path = destination / "adapter_config.json"
    metadata = json.loads(metadata_path.read_text())
    metadata["base_model_name_or_path"] = cfg["model_id"]
    metadata["revision"] = cfg["model_revision"]
    write_json(metadata_path, metadata)
    write_json(destination / "training_meta.json", {
        "successful_optimizer_updates": updates,
        "model_id": cfg["model_id"], "model_revision": cfg["model_revision"],
        "max_length": cfg["max_length"], "instruction": cfg["instruction"],
        "pooling": "attention-mask mean pooling in float32 + L2 normalization",
        "temperature": cfg["temperature"],
    })


@torch.inference_mode()
def encode_partition(encoder, texts, tokenizer, cfg, device, rank, world, log):
    indices = np.arange(rank, len(texts), world, dtype=np.int64)
    result = []
    cursor = 0
    batch_size = cfg["infer_batch_size"]
    next_report = 500
    while cursor < len(indices):
        batch_indices = indices[cursor:cursor + batch_size]
        try:
            features = tokenize([texts[i] for i in batch_indices], tokenizer, cfg, device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                embeddings = encoder(features)
            if not torch.isfinite(embeddings).all():
                raise RuntimeError("Нечисловые эмбеддинги при инференсе.")
            result.append(embeddings.float().cpu().numpy())
            cursor += len(batch_indices)
            del embeddings, features
        except torch.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            torch.cuda.empty_cache()
            log(f"Инференс: batch уменьшен до {batch_size}.")
            continue
        if cursor >= next_report:
            log(f"Инференс: обработано {cursor}/{len(indices)} текстов на GPU {rank}.")
            next_report += 500
    dimension = int(encoder.backbone.config.hidden_size)
    array = np.concatenate(result) if result else np.empty((0, dimension), dtype=np.float32)
    return indices, array


def predict(encoder, frame, tokenizer, cfg, device, rank, world, tag, log):
    encoder.eval()
    temporary = Path(cfg["job_dir"]) / "embedding_parts"
    temporary.mkdir(parents=True, exist_ok=True)
    queries = frame[["query_id", "query"]].drop_duplicates("query_id").reset_index(drop=True)
    documents = frame[["document_id", "document"]].drop_duplicates("document_id").reset_index(drop=True)
    for kind, texts in [
        ("query", [query_input(x, cfg) for x in queries["query"]]),
        ("document", documents["document"].tolist()),
    ]:
        indices, vectors = encode_partition(
            encoder, texts, tokenizer, cfg, device, rank, world, log
        )
        np.savez(temporary / f"{tag}_{kind}_{rank}.npz", indices=indices, vectors=vectors)
    if world > 1:
        dist.barrier()
    scores = None
    if rank == 0:
        merged = {}
        for kind, count in [("query", len(queries)), ("document", len(documents))]:
            full = None
            seen = np.zeros(count, dtype=bool)
            for other in range(world):
                path = temporary / f"{tag}_{kind}_{other}.npz"
                with np.load(path) as part:
                    index, vectors = part["indices"], part["vectors"]
                    if full is None:
                        full = np.zeros((count, vectors.shape[1]), dtype=np.float32)
                    if seen[index].any():
                        raise RuntimeError("Повторные индексы в частях инференса.")
                    full[index] = vectors
                    seen[index] = True
                path.unlink()
            if not seen.all():
                raise RuntimeError("Часть текстов пропущена при инференсе.")
            merged[kind] = full
        query_indices = pd.Index(queries["query_id"]).get_indexer(frame["query_id"])
        document_indices = pd.Index(documents["document_id"]).get_indexer(frame["document_id"])
        scores = np.einsum(
            "ij,ij->i", merged["query"][query_indices], merged["document"][document_indices]
        ).astype(np.float32)
        if not np.isfinite(scores).all():
            raise RuntimeError("Нечисловые предсказания.")
    if world > 1:
        dist.barrier()
    return scores


def run(cfg):
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world = int(os.environ.get("WORLD_SIZE", "1"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    if world > 1:
        dist.init_process_group("nccl", timeout=timedelta(minutes=45))
    started = time.monotonic()

    def log(message):
        if rank == 0:
            print(f'[{(time.monotonic() - started) / 60:.1f} мин | {cfg["job_name"]}] {message}', flush=True)

    seed = cfg["seed"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_path"], trust_remote_code=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    validation = None
    if cfg.get("validation_csv"):
        validation = pd.read_csv(cfg["validation_csv"], dtype=str, keep_default_na=False)
        validation["relevance"] = pd.to_numeric(validation["relevance"])
    test = pd.read_csv(cfg["test_csv"], dtype=str, keep_default_na=False)
    dataset = Triplets(cfg["triplets_jsonl"])
    if len(dataset) < world:
        raise RuntimeError("Недостаточно обучающих примеров.")
    sampler = DistributedSampler(
        dataset, num_replicas=world, rank=rank, shuffle=True, seed=seed, drop_last=False
    ) if world > 1 else None
    loader = DataLoader(
        dataset, batch_size=cfg["train_batch_size"], sampler=sampler,
        shuffle=sampler is None, collate_fn=identity_collate,
        num_workers=0, drop_last=False,
    )
    encoder = load_backbone(cfg, device)
    trainable = [p for p in encoder.parameters() if p.requires_grad]
    if not trainable or any(p.dtype != torch.float32 for p in trainable):
        raise RuntimeError("LoRA должна иметь обучаемые параметры float32 для FP16 AMP.")
    log(f"Обучаемых параметров: {sum(p.numel() for p in trainable):,}; GPU: {world}.")
    log(f"Примеров: {len(dataset)}; шагов на GPU за эпоху: {len(loader)}.")

    base_score = None
    if validation is not None:
        log("Оценка исходной модели на валидации.")
        before = predict(encoder, validation, tokenizer, cfg, device, rank, world, "base", log)
        if rank == 0:
            baseline = validation[["id", "query_id", "relevance"]].copy()
            baseline["prediction_base"] = before
            base_score = ndcg10(baseline, "prediction_base")
            baseline.to_csv(Path(cfg["job_dir"]) / "validation_base.csv", index=False)
            log(f"NDCG@10 до обучения: {base_score:.6f}")

    training = DistributedDataParallel(
        encoder, device_ids=[local_rank], output_device=local_rank,
        find_unused_parameters=False, broadcast_buffers=False,
    ) if world > 1 else encoder
    optimizer = torch.optim.AdamW(trainable, lr=cfg["learning_rate"], weight_decay=0.01)
    updates_per_epoch = math.ceil(len(loader) / cfg["gradient_accumulation"])
    total_updates = updates_per_epoch * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(total_updates * 0.1)),
        num_training_steps=total_updates,
    )
    scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
    updates, skipped = 0, 0
    best_score = -float("inf")
    best_path = Path(cfg["job_dir"]) / "best_adapter"
    epoch_metrics = []
    optimizer.zero_grad(set_to_none=True)
    recent_losses = []
    for epoch in range(cfg["epochs"]):
        if sampler is not None:
            sampler.set_epoch(epoch)
        training.train()
        for micro, rows in enumerate(loader):
            window_start = (micro // cfg["gradient_accumulation"]) * cfg["gradient_accumulation"]
            window_size = min(cfg["gradient_accumulation"], len(loader) - window_start)
            synchronized = (micro + 1) % cfg["gradient_accumulation"] == 0 or micro + 1 == len(loader)
            texts = []
            for row in rows:
                if len(row["negatives"]) != cfg["hard_negatives"]:
                    raise RuntimeError("Неверное число hard negatives.")
                if row["positive_id"] in row["negative_ids"]:
                    raise RuntimeError("Положительный документ попал в отрицательные.")
                texts.extend([query_input(row["query"], cfg), row["positive"], *row["negatives"]])
            features = tokenize(texts, tokenizer, cfg, device)
            context = training.no_sync() if world > 1 and not synchronized else contextlib.nullcontext()
            with context:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    embeddings = training(features).reshape(
                        len(rows), 2 + cfg["hard_negatives"], -1
                    )
                    logits = torch.einsum(
                        "bd,bnd->bn", embeddings[:, 0], embeddings[:, 1:]
                    ).float() / cfg["temperature"]
                    loss = F.cross_entropy(
                        logits, torch.zeros(len(rows), dtype=torch.long, device=device)
                    )
                if not torch.isfinite(loss):
                    raise RuntimeError("Loss стал NaN/Inf: обучение остановлено.")
                scaler.scale(loss / window_size).backward()
            recent_losses.append(float(loss.detach()))
            del embeddings, logits, features, loss
            if not synchronized:
                continue
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            successful = scaler.get_scale() >= previous_scale
            optimizer.zero_grad(set_to_none=True)
            if not successful:
                skipped += 1
                log(f"Переполнение FP16: пропущен шаг, всего {skipped}.")
                if skipped > 8:
                    raise RuntimeError("Слишком много пропусков FP16. Полный прогон остановлен.")
                continue
            updates += 1
            scheduler.step()
            if updates == 1 or updates % cfg["log_every"] == 0:
                memory = torch.cuda.max_memory_allocated(device) / 1024**3
                log(
                    f"эпоха {epoch + 1}/{cfg['epochs']}, обновление {updates}/{total_updates}, "
                    f"loss={np.mean(recent_losses):.4f}, память GPU0={memory:.2f} GiB"
                )
                recent_losses.clear()
            if updates == cfg["smoke_updates"]:
                if rank == 0:
                    changed = any(
                        "lora_B" in name and torch.count_nonzero(parameter.detach()).item() > 0
                        for name, parameter in encoder.named_parameters() if parameter.requires_grad
                    )
                    if not changed:
                        raise RuntimeError("LoRA-B осталась нулевой: обновления не подтверждены.")
                    smoke = Path(cfg["job_dir"]) / "smoke_adapter"
                    save_adapter(encoder, smoke, cfg, updates)
                    on_disk = load_file(str(smoke / "adapter_model.safetensors"))
                    in_memory = get_peft_model_state_dict(encoder.backbone)
                    if set(on_disk) != set(in_memory):
                        raise RuntimeError("Не совпадают ключи сохранённого адаптера.")
                    if not all(torch.equal(on_disk[k], v.detach().cpu()) for k, v in in_memory.items()):
                        raise RuntimeError("Сохранённый адаптер отличается от обученного.")
                    del on_disk, in_memory
                    log("Короткая проверка пройдена: градиенты обновляют LoRA; адаптер сохранён и прочитан.")
                if world > 1:
                    dist.barrier()
            if updates % cfg["save_every"] == 0:
                if rank == 0:
                    checkpoint = Path(cfg["job_dir"]) / "checkpoints" / f"step_{updates:06d}"
                    save_adapter(encoder, checkpoint, cfg, updates)
                    checkpoints = sorted(checkpoint.parent.glob("step_*"))
                    for old in checkpoints[:-2]:
                        shutil.rmtree(old)
                if world > 1:
                    dist.barrier()

        if updates == 0:
            raise RuntimeError("Ни одного успешного обновления весов.")
        score = None
        if validation is not None:
            prediction = predict(
                encoder, validation, tokenizer, cfg, device, rank, world,
                f"epoch_{epoch + 1}", log,
            )
            if rank == 0:
                evaluated = validation[["id", "query_id", "relevance"]].copy()
                evaluated["prediction"] = prediction
                score = ndcg10(evaluated)
                evaluated.to_csv(Path(cfg["job_dir"]) / f"validation_epoch_{epoch + 1}.csv", index=False)
                log(f"NDCG@10 после эпохи {epoch + 1}: {score:.6f}")
        if rank == 0:
            epoch_metrics.append({"epoch": epoch + 1, "ndcg10": score, "updates": updates})
            if validation is None or score > best_score:
                best_score = score if score is not None else 0.0
                save_adapter(encoder, best_path, cfg, updates)
                write_json(Path(cfg["job_dir"]) / "best.json", {
                    "epoch": epoch + 1, "ndcg10": score, "updates": updates,
                })
        if world > 1:
            dist.barrier()

    log("Повторная загрузка выбранного обученного адаптера.")
    del training, encoder, optimizer, scheduler, scaler, trainable
    gc.collect()
    torch.cuda.empty_cache()
    encoder = load_backbone(cfg, device, adapter_path=str(best_path), trainable=False)
    reloaded_score = None
    if validation is not None:
        prediction = predict(encoder, validation, tokenizer, cfg, device, rank, world, "reloaded", log)
        if rank == 0:
            evaluated = validation[["id", "query_id", "relevance"]].copy()
            evaluated["prediction"] = prediction
            evaluated["fold"] = cfg["fold"]
            baseline = pd.read_csv(Path(cfg["job_dir"]) / "validation_base.csv", dtype={"id": str})
            evaluated = evaluated.merge(
                baseline[["id", "prediction_base"]], on="id", how="left", validate="one_to_one"
            )
            reloaded_score = ndcg10(evaluated)
            selected = json.loads((Path(cfg["job_dir"]) / "best.json").read_text())
            if abs(reloaded_score - selected["ndcg10"]) > 0.002:
                raise RuntimeError("Заметное изменение NDCG после загрузки адаптера.")
            evaluated.to_csv(Path(cfg["job_dir"]) / "validation_predictions.csv", index=False)
            log(f"NDCG@10 загруженного адаптера: {reloaded_score:.6f}")

    prediction = predict(encoder, test, tokenizer, cfg, device, rank, world, "test", log)
    if rank == 0:
        pd.DataFrame({"id": test["id"], "prediction": prediction}).to_csv(
            Path(cfg["job_dir"]) / "test_predictions.csv", index=False
        )
        write_json(Path(cfg["job_dir"]) / "metrics.json", {
            "job": cfg["job_name"], "fold": cfg["fold"],
            "ndcg10_base": base_score, "ndcg10_lora": reloaded_score,
            "successful_optimizer_updates": updates, "fp16_skipped_updates": skipped,
            "epochs": epoch_metrics, "minutes": (time.monotonic() - started) / 60,
        })
        log("Готово: адаптер, метрики, validation_predictions.csv и test_predictions.csv сохранены.")
    if world > 1:
        dist.barrier()
        dist.destroy_process_group()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    run(json.loads(Path(args.config).read_text(encoding="utf-8")))
''', encoding='utf-8')

In [ ]:
subprocess.run([
    sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", "--max_restarts=0",
    "--nproc_per_node=2", str(SCRIPTS / "giga_worker.py"), "--config", str(CONFIG_FILE),
], env=worker_env, check=True)

metrics = json.loads((FOLD / "metrics.json").read_text(encoding="utf-8"))
print(f"Обновлений LoRA {metrics['successful_optimizer_updates']}. "
      f"Адаптер сохранён в {FOLD / 'best_adapter'}.")

## Ранжирование и файл отправки




In [ ]:
(SCRIPTS / 'base_worker.py').write_text(r'''import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from transformers import AutoModel, AutoTokenizer


def encode_batch(batch, tokenizer, model, max_length):
    features = tokenizer(batch, padding=True, truncation=True, max_length=max_length,
                         return_tensors="pt", return_token_type_ids=False).to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        hidden = model(**features).last_hidden_state.float()
        mask = features["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
        return F.normalize(pooled, p=2, dim=-1).cpu().numpy()


@torch.inference_mode()
def encode(texts, tokenizer, model, cfg, rank, name):
    vectors = []
    batch_size = cfg["infer_batch_size"]
    cursor = 0
    next_report = 1000
    while cursor < len(texts):
        batch = texts[cursor:cursor + batch_size]
        try:
            vectors.append(encode_batch(batch, tokenizer, model, cfg["max_length"]))
        except torch.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f"GPU {rank} уменьшает batch до {batch_size}", flush=True)
        else:
            cursor += len(batch)
            if cursor >= next_report or cursor == len(texts):
                print(f"GPU {rank} · {name} {cursor}/{len(texts)}", flush=True)
                next_report += 1000
    return np.concatenate(vectors)


def score_pairs(rows, tokenizer, model, cfg, rank):
    queries = rows[["query_id", "query"]].drop_duplicates("query_id").set_index("query_id")
    docs = rows[["document_id", "document"]].drop_duplicates("document_id").set_index("document_id")
    q_texts = [f"Instruct: {cfg['instruction']}\nQuery: {text}" for text in queries["query"]]
    q = encode(q_texts, tokenizer, model, cfg, rank, "запросы")
    d = encode(docs["document"].tolist(), tokenizer, model, cfg, rank, "фрагменты")
    q_idx = queries.index.get_indexer(rows["query_id"])
    d_idx = docs.index.get_indexer(rows["document_id"])
    return np.einsum("ij,ij->i", q[q_idx], d[d_idx])


def main(directory, rank):
    cfg = json.loads((directory / "worker_config.json").read_text())
    torch.cuda.set_device(rank)
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_path"], trust_remote_code=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained(cfg["model_path"], trust_remote_code=True,
                                      dtype=torch.float16, attn_implementation="sdpa")
    model.config.use_cache = False
    model.to("cuda").eval()
    rows = pd.read_csv(directory / f"input_{rank}.csv", dtype=str, keep_default_na=False)
    scores = score_pairs(rows, tokenizer, model, cfg, rank)
    if not np.isfinite(scores).all():
        raise ValueError("Модель вернула нечисловые оценки")
    rows[["id"]].assign(prediction_base=scores).to_csv(directory / f"scores_{rank}.csv", index=False)


if __name__ == "__main__":
    main(Path(sys.argv[1]), int(sys.argv[2]))
''', encoding='utf-8')

In [ ]:
(SCRIPTS / 'blend.py').write_text(r'''import json
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd


def read_pairs(path, columns):
    data = pd.read_csv(path, usecols=columns, dtype=str, keep_default_na=False)
    if data.empty or data["id"].duplicated().any() or data["id"].str.strip().eq("").any():
        raise ValueError(f"Проверь число строк и уникальность id в {path}")
    return data


def ordered_scores(rows, prediction_data, column):
    indexed = prediction_data.set_index("id", verify_integrity=True)
    expected = pd.Index(rows["id"])
    if len(indexed) != len(expected) or not expected.isin(indexed.index).all():
        raise ValueError(f"Состав id различается для {column}")
    values = pd.to_numeric(indexed.loc[expected, column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"В {column} есть пропущенные или бесконечные оценки")
    return values


def query_z(values, groups):
    s = pd.Series(np.asarray(values, dtype=float), index=groups.index)
    grouped = s.groupby(groups, sort=False)
    scale = grouped.transform("std", ddof=0)
    return ((s - grouped.transform("mean")) / scale.replace(0, np.nan)).fillna(0)


def blend(base, lora, groups):
    return (query_z(base, groups) + query_z(lora, groups)) / 2


def ndcg10(frame, score):
    ranking = frame[["id", "query_id", "relevance"]].assign(score=np.asarray(score, dtype=float))
    ranking = ranking.sort_values(["query_id", "score", "id"], ascending=[True, False, True])
    pos = ranking.groupby("query_id", sort=False).cumcount().to_numpy()
    ranking["dcg"] = np.where(pos < 10, ranking["relevance"] / np.log2(pos + 2), 0)
    grouped = ranking.groupby("query_id")
    discounts = np.r_[0, np.cumsum(1 / np.log2(np.arange(2, 12)))]
    ideal = discounts[grouped["relevance"].sum().clip(upper=10).to_numpy(dtype=int)]
    dcg = grouped["dcg"].sum().to_numpy()
    return float(np.divide(dcg, ideal, out=np.zeros_like(dcg), where=ideal > 0).mean())


def run_workers(rows, worker_cfg, n_gpu):
    with tempfile.TemporaryDirectory(prefix="giga_base_") as tmp:
        job = Path(tmp)
        script = Path(__file__).with_name("base_worker.py")
        (job / "worker_config.json").write_text(json.dumps(worker_cfg, ensure_ascii=False), encoding="utf-8")
        # Все пары одного запроса обрабатывает один процесс.
        qids = rows["query_id"].drop_duplicates().tolist()
        processes = []
        env = dict(os.environ, OMP_NUM_THREADS="2", TOKENIZERS_PARALLELISM="false")
        workers = min(n_gpu, len(qids))
        try:
            for rank in range(workers):
                shard = rows.loc[rows["query_id"].isin(qids[rank::workers])]
                shard.to_csv(job / f"input_{rank}.csv", index=False)
                processes.append(subprocess.Popen([sys.executable, "-u", str(script), str(job), str(rank)], env=env))
            while True:
                codes = [p.poll() for p in processes]
                if any(code not in (0, None) for code in codes):
                    raise RuntimeError("Сбой в процессе инференса. Подробности выше в логе GPU.")
                if all(code == 0 for code in codes):
                    break
                time.sleep(1)
        finally:
            for p in processes:
                if p.poll() is None:
                    p.terminate()
            for p in processes:
                p.wait()
        pieces = [read_pairs(job / f"scores_{rank}.csv", ["id", "prediction_base"]) for rank in range(workers)]
        return ordered_scores(rows, pd.concat(pieces, ignore_index=True), "prediction_base")


def save_csv(frame, path):
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".tmp", delete=False) as handle:
        tmp = Path(handle.name)
    try:
        frame.to_csv(tmp, index=False)
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)


def main(config_file, output):
    cfg = json.loads(Path(config_file).read_text(encoding="utf-8"))
    fold = Path(cfg["job_dir"])
    inputs = ["id", "query_id", "document_id", "query", "document"]
    test = read_pairs(cfg["test_csv"], inputs)
    val = read_pairs(fold / "validation_predictions.csv",
                     ["id", "query_id", "relevance", "prediction_base", "prediction"])
    val["relevance"] = pd.to_numeric(val["relevance"], errors="raise")
    valid_base = ordered_scores(val, val, "prediction_base")
    valid_lora = ordered_scores(val, val, "prediction")
    valid_blend = blend(valid_base, valid_lora, val["query_id"])
    print(f"Фолд 0 · {val['query_id'].nunique()} запросов · NDCG@10 "
          f"Giga {ndcg10(val, valid_base):.6f}, LoRA {ndcg10(val, valid_lora):.6f}, "
          f"смесь {ndcg10(val, valid_blend):.6f}", flush=True)

    # Сохраняем порядок батчей исходного расчёта, включая 24 контрольные пары.
    controls = val.sample(n=min(24, len(val)), random_state=993)[["id", "prediction_base"]]
    controls = controls.merge(read_pairs(cfg["validation_csv"], inputs), on="id", validate="one_to_one")
    if len(controls) != min(24, len(val)) or controls["id"].isin(test["id"]).any():
        raise ValueError("Неверный набор контрольных id")
    pairs = pd.concat([test, controls[inputs]], ignore_index=True)
    worker_cfg = {k: cfg[k] for k in ("model_path", "instruction", "max_length", "infer_batch_size")}
    scores = run_workers(pairs, worker_cfg, n_gpu=2)
    error = np.max(np.abs(scores[len(test):] - controls["prediction_base"].to_numpy(dtype=float)))
    if error > 0.02:
        raise ValueError(f"Расхождение оценок исходной модели на контрольных парах {error:.6f}")
    lora = ordered_scores(test, read_pairs(fold / "test_predictions.csv", ["id", "prediction"]), "prediction")
    final = blend(scores[:len(test)], lora, test["query_id"])
    submission = test[["id"]].assign(prediction=final.to_numpy())
    if len(test) != 13507 or test["query_id"].nunique() != 1305 or not np.isfinite(final).all():
        raise ValueError("Неверное число тестовых пар, запросов или нечисловые предсказания")
    save_csv(submission, output)
    print(f"Файл для Kaggle {output} · {len(submission)} пар · контрольное расхождение {error:.6f}")


if __name__ == "__main__":
    main(sys.argv[1], Path(sys.argv[2]))
''', encoding='utf-8')

submission_path = OUTPUT / "submission.csv"
subprocess.run([sys.executable, "-u", str(SCRIPTS / "blend.py"), str(CONFIG_FILE), str(submission_path)],
               env=worker_env, check=True)